# FastSpeech 2: English Text-to-Speech Training & Evaluation
### Assignment: FastSpeech 2-Based TTS (Svarah English Dataset)

This notebook trains FastSpeech 2 on English audio, validates on separate splits, evaluates the 5 test sentences (E1–E4), generates variance adaptor plots, and packages all deliverables into a downloadable zip file.

## 1. Setup Environment & Check GPU

In [ ]:
!nvidia-smi
!pip install -q g2p_en pyworld librosa soundfile matplotlib pyyaml tqdm torch torchaudio

## 2. Unpack Project Code from Input

In [ ]:
import os, glob, zipfile, shutil

os.makedirs('/kaggle/working/project', exist_ok=True)

# Look for uploaded zip or extracted folder in /kaggle/input
zip_files = glob.glob('/kaggle/input/**/FastSpeech2_Project_Source.zip', recursive=True) + glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zip_files:
    with zipfile.ZipFile(zip_files[0], 'r') as z:
        z.extractall('/kaggle/working/project')
    print('Successfully unpacked zip:', zip_files[0])
else:
    for d in glob.glob('/kaggle/input/*'):
        if os.path.isdir(d):
            for item in os.listdir(d):
                s = os.path.join(d, item)
                dest = os.path.join('/kaggle/working/project', item)
                if os.path.isdir(s):
                    shutil.copytree(s, dest, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, dest)
            print('Copied files from:', d)

# Ensure output directories exist
os.makedirs('/kaggle/working/project/English/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/test_generated_audios', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/duration', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/pitch', exist_ok=True)
os.makedirs('/kaggle/working/project/Results/English/variance_adaptor_plots/energy', exist_ok=True)
print('Project ready. Contents:', os.listdir('/kaggle/working/project'))

## 3. Dataset Setup (Svarah English - 5h Train / 1h Val)

In [ ]:
os.makedirs('/kaggle/working/project/English/data/Svarah/wavs', exist_ok=True)
print('Svarah English data directories initialized.')

## 4. Run Preprocessing Pipeline

In [ ]:
%cd /kaggle/working/project/English
!python preprocess.py config.yaml

## 5. Train FastSpeech 2 Model

In [ ]:
%cd /kaggle/working/project/English
!python train.py config.yaml

## 6. Run Inference & Generate Variance Plots for Test Sentences (E1–E4)

In [ ]:
%cd /kaggle/working/project/English
!python inference.py config.yaml checkpoints/best_model.pth

## 7. Package Deliverables for Download

In [ ]:
%cd /kaggle/working
!zip -r english_results.zip project/Results/English project/English/checkpoints
print('english_results.zip is ready for download in Kaggle Output!')